# TabNet and trust research on BRFSS
Exploratory development only. No reuse of the previously inspected test set for tuning. These experiments do not provide clinical recommendations. Set a GPU runtime for TabNet if available. Mount your Drive copy; raw data never enter git.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
uploaded=files.upload()
assert "research-baseline.patch" in uploaded


In [ ]:
from pathlib import Path
import subprocess,sys
project=Path('/content/BRFSS-Stroke-Neural-Feature-Learning')
subprocess.run(['git','clone','https://github.com/DenisParker7/BRFSS-Stroke-Neural-Feature-Learning.git',str(project)],check=True)
subprocess.run(['git','checkout','095ae4d367d1c34000e1c235e9f4a93596c99791'],cwd=project,check=True)
subprocess.run(['git','switch','-c','research/tabnet-trust'],cwd=project,check=True)
subprocess.run(['git','apply','/content/research-baseline.patch'],cwd=project,check=True)


In [ ]:
# Colab normally provides a GPU-capable torch build; preserve it.
import torch
print('torch',torch.__version__,'GPU',torch.cuda.is_available())
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-tabnet.txt'],cwd=project,check=True)


In [ ]:
data=Path('/content/drive/MyDrive/LLCP2024.XPT')  # Set your actual path.
assert data.is_file()
output=Path('/content/drive/MyDrive/StrokeResearch/tabnet-trust-v1')
output.mkdir(parents=True,exist_ok=False)


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=project,check=True)


In [ ]:
subprocess.run([sys.executable,'-m','truststroke.prepare_development','--data',str(data),'--output',str(output/'protocol')],cwd=project,check=True)


In [ ]:
subprocess.run([sys.executable,'-m','truststroke.tabnet_search','--data',str(data),'--partitions-from',str(output/'protocol'),'--output',str(output/'tabnet'),'--epochs','30'],cwd=project,check=True)


In [ ]:
subprocess.run([sys.executable,'-m','truststroke.trust_experiment','--data',str(data),'--partitions-from',str(output/'protocol'),'--output',str(output/'trust'),'--epochs','20'],cwd=project,check=True)


In [ ]:
subprocess.run([sys.executable,'-m','truststroke.attribution_convergence','--data',str(data),'--run',str(output/'trust')],cwd=project,check=True)


The original CPU runs and observed negative findings are documented in docs/TABNET_TRUST_RESULTS.md. Colab installation and Drive integration are not executed by local notebook syntax checks. A GPU/build change may change numerical results. Do not keep searching against an external cohort after inspecting its scores.